# Module 11: exercises  (solutions)

These are small functions you'll reuse in your project's report and pipeline. The hands-on testing exercise lives in `exercise/`.

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

## Exercise 1

**A sentence from numbers.** Write `describe_difference(label_a, label_b, mean_a, mean_b, ci_low, ci_high, p, unit)` returning one sentence of the form

`On average, {label_a} had {X} {unit} {higher|lower} than {label_b} (95% CI {low} to {high}; {p_text}).`

where `X` is the absolute difference `mean_a - mean_b` to **1 decimal**, the CI bounds have 1 decimal, and `p_text` is `p < 0.001` when p is below 0.001, otherwise `p = ` followed by 3 decimals.
If the interval contains 0, append the sentence ` The data are compatible with no difference, so the result is inconclusive.`

In [ ]:
import inspect

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def describe_difference(label_a, label_b, mean_a, mean_b, ci_low, ci_high, p, unit):
    diff = mean_a - mean_b
    direction = "higher" if diff > 0 else "lower"
    p_text = "p < 0.001" if p < 0.001 else f"p = {p:.3f}"
    text = (f"On average, {label_a} had {abs(diff):.1f} {unit} {direction} than {label_b} "
            f"(95% CI {ci_low:.1f} to {ci_high:.1f}; {p_text}).")
    if ci_low <= 0 <= ci_high:
        text += " The data are compatible with no difference, so the result is inconclusive."
    return text

In [ ]:
s1 = describe_difference("group A", "group B", 70.0, 66.0, 1.2, 6.8, 0.0049, "points")
assert s1 == "On average, group A had 4.0 points higher than group B (95% CI 1.2 to 6.8; p = 0.005).", s1
s2 = describe_difference("A", "B", 66.0, 68.9, -6.1, 0.3, 0.0724, "points")
assert "2.9 points lower than B" in s2 and "p = 0.072" in s2 and s2.endswith("so the result is inconclusive."), s2
s3 = describe_difference("A", "B", 5.0, 1.0, 3.0, 5.0, 0.0000004, "min")
assert "p < 0.001" in s3 and "inconclusive" not in s3, s3
print("✅ Correct!")
print(s2)

## Exercise 2

**A chart that makes a point.** Write `takeaway_chart(rides)` that draws a bar chart of the **average rides per day for weekdays vs weekends** and returns the Axes, with:

* a **title that states the takeaway with the number**, e.g. "Weekends see 11% fewer rides than weekdays" (compute the percentage from the data: use the absolute percent difference rounded to a whole number, followed by `%`);
* y-axis starting at **0** and a y-label that contains the word `rides`;
* the top and right spines removed (`sns.despine` or `ax.spines[...].set_visible(False)`).

In [ ]:
import seaborn as sns

rides = pd.read_csv(DATA / "daily_rides.csv", parse_dates=["date"])

def takeaway_chart(rides):
    fig, ax = plt.subplots(figsize=(5, 3.5))
    means = rides.groupby("is_weekend")["rides"].mean()
    change = (means[1] - means[0]) / means[0] * 100
    word = "fewer" if change < 0 else "more"
    ax.bar(["Weekdays", "Weekends"], [means[0], means[1]], color=["#bbbbbb", "#0072B2"])
    ax.set_title(f"Weekends see {abs(change):.0f}% {word} rides than weekdays", loc="left")
    ax.set_ylabel("Average rides per day")
    ax.set_ylim(0, None)
    sns.despine(ax=ax)
    return ax

In [ ]:
ax = takeaway_chart(rides)
means = rides.groupby("is_weekend")["rides"].mean()
pct = abs((means[1] - means[0]) / means[0] * 100)
title = " ".join(ax.get_title(loc=where) for where in ("left", "center", "right")).strip()   # titles can be left-aligned
assert f"{pct:.0f}%" in title, f"The title should contain the number {pct:.0f}%: {title!r}"
assert ax.get_ylim()[0] == 0, "Bars should start at zero"
assert "rides" in ax.get_ylabel().lower(), "y-label should say what is measured (rides)"
assert not ax.spines["top"].get_visible() and not ax.spines["right"].get_visible(), "Remove the top/right spines"
print("✅ Correct!", title)

## Exercise 3

**Write the docstring.** Rewrite `weekend_effect` so it has (a) a **Google-style docstring** with `Args:` and `Returns:` sections (and a sentence mentioning *percent*), and (b) **type hints** for the argument and the return value.
It must still return the percent change in mean rides on weekends relative to weekdays.

In [ ]:
def weekend_effect(df: pd.DataFrame) -> float:
    '''Return the percent change in mean daily rides on weekends relative to weekdays.

    Args:
        df: A DataFrame with a numeric ``rides`` column and an ``is_weekend`` column (1 = weekend, 0 = weekday).

    Returns:
        ((weekend mean - weekday mean) / weekday mean) * 100. A negative value means weekends are quieter.
    '''
    weekend = df.loc[df["is_weekend"] == 1, "rides"].mean()
    weekday = df.loc[df["is_weekend"] == 0, "rides"].mean()
    return float((weekend - weekday) / weekday * 100)

In [ ]:
doc = inspect.getdoc(weekend_effect) or ""
assert len(doc) > 60, "Write a descriptive docstring"
assert "Args:" in doc and "Returns:" in doc, "Use Google style: 'Args:' and 'Returns:' sections"
assert "percent" in doc.lower(), "Say that the result is a percent change"
assert {"df", "return"} <= set(weekend_effect.__annotations__), "Add type hints for df and the return value"
assert abs(weekend_effect(rides) - (-10.8)) < 0.1
print("✅ Correct!")

## Exercise 4

**Validate the data.** The cell below prepares `raw` (the messy orders) and `clean` (after Module 5's cleaning). Write `validate_orders(df)` returning a **list of problem descriptions** (strings), empty when the table is fine. Check that:
`order_id` has no duplicates; every `city` is one of `Bangkok`, `Chiang Mai`, `Phuket`; `unit_price` is numeric with no missing values and all positive; and `order_date` has a datetime type.
It should return `[]` for `clean` and at least 3 problems for `raw`.

In [ ]:
raw = pd.read_csv(DATA / "messy_orders.csv")

def clean_orders(raw):
    df = raw.drop_duplicates().reset_index(drop=True)
    for col in ["customer", "city", "item"]:
        df[col] = df[col].str.strip().str.title()
    df["city"] = df["city"].replace({"Chiangmai": "Chiang Mai"})
    df["unit_price"] = pd.to_numeric(df["unit_price"].str.replace("$", "", regex=False), errors="coerce")
    df["order_date"] = pd.to_datetime(df["order_date"], format="mixed")
    df["unit_price"] = df["unit_price"].fillna(df["item"].map(df.groupby("item")["unit_price"].median()))
    df.loc[df["quantity"] > 50, "quantity"] = np.nan
    df["quantity"] = df["quantity"].fillna(df["quantity"].median())
    return df

clean = clean_orders(raw)

def validate_orders(df):
    problems = []
    if df["order_id"].duplicated().any():
        problems.append("duplicate order_id values")
    if not set(df["city"].unique()) <= {"Bangkok", "Chiang Mai", "Phuket"}:
        problems.append("unexpected city names")
    price = pd.to_numeric(df["unit_price"], errors="coerce")
    if not pd.api.types.is_numeric_dtype(df["unit_price"]):
        problems.append("unit_price is not numeric")
    if price.isna().any():
        problems.append("missing or non-numeric unit_price values")
    elif (price <= 0).any():
        problems.append("non-positive unit_price values")
    if not pd.api.types.is_datetime64_any_dtype(df["order_date"]):
        problems.append("order_date is not a datetime column")
    return problems

In [ ]:
assert validate_orders(clean) == [], validate_orders(clean)
found = validate_orders(raw)
assert isinstance(found, list) and all(isinstance(p, str) for p in found)
assert len(found) >= 3, f"The raw table has several problems; you reported {found}"
bad = clean.copy(); bad.loc[0, "unit_price"] = -1
assert validate_orders(bad), "A negative price should be flagged"
print("✅ Correct! Problems in the raw table:", found)

## Exercise 5

**Protect small groups.** Write `suppress_small_groups(counts, min_n=5)` that returns a copy of the Series with every count below `min_n` replaced by `NaN` (and never modifies its input).

In [ ]:
def suppress_small_groups(counts, min_n=5):
    out = counts.astype(float).copy()
    out[out < min_n] = np.nan
    return out

In [ ]:
original = pd.Series({"a": 20, "b": 4, "c": 5, "d": 1})
snapshot = original.copy()
out = suppress_small_groups(original)
assert out["a"] == 20 and out["c"] == 5, "counts at or above min_n are kept"
assert np.isnan(out["b"]) and np.isnan(out["d"]), "counts below min_n become NaN"
assert original.equals(snapshot), "do not modify the input"
assert suppress_small_groups(original, min_n=10)["a"] == 20 and np.isnan(suppress_small_groups(original, min_n=25)["a"])
print("✅ Correct!")

## Exercise 6

**Record the environment.** Write `environment_report()` returning a dict with the keys `"python"`, `"numpy"` and `"pandas"` whose values are version strings
(use `sys.version.split()[0]` and `importlib.metadata.version`).

In [ ]:
import importlib.metadata as md
import sys

def environment_report():
    return {
        "python": sys.version.split()[0],
        "numpy": md.version("numpy"),
        "pandas": md.version("pandas"),
    }

In [ ]:
rep = environment_report()
assert set(rep) == {"python", "numpy", "pandas"}, rep
assert rep["python"].startswith("3.") and all(isinstance(v, str) and v for v in rep.values())
assert rep["pandas"] == pd.__version__ and rep["numpy"] == np.__version__
print("✅ Correct!", rep)

## Exercise 7

**Choose the best sentence.** Which is the best way to report the result of a study? Set `answer` to a letter.

* **a)** "The treatment worked (p < 0.05)."
* **b)** "We found a significant effect."
* **c)** "Patients on the new schedule slept 22 minutes longer per night on average (95% CI 6 to 38; n = 120, randomised), a modest improvement; the trial lasted only 4 weeks, so long-term effects are unknown."
* **d)** "The new schedule increases sleep, proven by our analysis."

In [ ]:
answer = "c"

In [ ]:
assert answer == "c", "Look for: effect size in units, an interval, the sample/design, and a stated limitation"
print("✅ Correct!")